# MSC Figure 6 — Tushev isoform analysis (corrected, reviewer-ready)

This notebook analyzes the PASS-defined 3′UTR isoforms from Tushev et al. (Neuron, 2018) as a third external dataset for the Material-Selective Capture (MSC) manuscript.

## Required files

Upload the original supplementary workbooks and extracted CSV sheets under:

```text
MyDrive/MSC_Results_0726/figure6_tushev_isoform/inputs/
├── paper/
├── supplementary_original/
│   ├── 1-s2.0-S0896627318302368-mmc2.xls
│   ├── 1-s2.0-S0896627318302368-mmc3.xls
│   ├── 1-s2.0-S0896627318302368-mmc4.xls
│   └── 1-s2.0-S0896627318302368-mmc5.xls
└── supplementary_csv/
    ├── 1-s2.0-S0896627318302368-mmc2-PASSData.csv
    ├── 1-s2.0-S0896627318302368-mmc2-Header.csv
    ├── 1-s2.0-S0896627318302368-mmc2-Description.csv
    └── all other extracted CSV sheets
```

The MSC files must be present at:

```text
MyDrive/MSC_Results_0726/msc_model_feedback.py
MyDrive/MSC_Results_0726/config_msc_feedback.yaml
```

## Locked design

- Unit: experimentally defined PASS 3′UTR isoform.
- Universe: neuron-enriched, significantly neuropil- or somata-localized isoforms.
- `upstream.length` is reconstructed from a unique rn5 RefGene stop boundary in the PASS direction (`fwdgroup` or `revgroup`). Genes with ambiguous stop boundaries are excluded rather than guessed.
- Genes represented in both localization groups are reserved for a within-gene isoform analysis.
- Primary panel: one isoform per gene, 250 neuropil plus 250 somata isoforms, jointly matched on tissue expression and 3′UTR length.
- No G/rG4 feature is calculated before panel freezing.
- Primary PSD calibration: relaxation time, 100 seeds. Elastic modulus and mobile fraction: 30 seeds each.
- Outputs use the same schema as the Middleton and Perez plotting tables.

In [ ]:
# 0. Runtime-local setup, dependencies and required uploads
# Drive is NOT required. Upload the three input files here, at the very start.
%pip -q install pandas numpy scipy statsmodels pyyaml openpyxl xlrd pyfaidx requests

from pathlib import Path
import os, re, sys, gzip, json, yaml, shutil, hashlib, datetime, zipfile, importlib, warnings
import requests
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import milp, LinearConstraint, Bounds, linear_sum_assignment
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from google.colab import files
warnings.filterwarnings('ignore')


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def place(source, destination):
    """Move an uploaded file to its canonical path, tolerating identical paths."""
    source, destination = Path(source), Path(destination)
    if source.resolve() == destination.resolve():
        return
    shutil.copy2(source, destination)


UPLOAD_DIR = Path('/content')
MODEL_PATH = UPLOAD_DIR / 'msc_model_feedback.py'
CONFIG_PATH = UPLOAD_DIR / 'config_msc_feedback.yaml'

# The PASSData supplementary table must contain 'mmc2' and 'PASSData' in its filename.
def find_passdata(directory):
    hits = [
        path for path in Path(directory).glob('*.csv')
        if 'mmc2' in path.name.lower() and 'passdata' in path.name.lower()
    ]
    return hits


passdata_hits = find_passdata(UPLOAD_DIR)

if not (MODEL_PATH.exists() and CONFIG_PATH.exists() and len(passdata_hits) == 1):
    print('Upload these three files now:')
    print('  1) msc_model_feedback.py')
    print('  2) config_msc_feedback.yaml')
    print('  3) the Tushev mmc2 PASSData CSV, e.g.')
    print('     1-s2.0-S0896627318302368-mmc2-PASSData.csv')
    uploaded = files.upload()
    for name in uploaded:
        source = UPLOAD_DIR / name
        if name.endswith('.py'):
            place(source, MODEL_PATH)
        elif name.endswith(('.yaml', '.yml')):
            place(source, CONFIG_PATH)
    passdata_hits = find_passdata(UPLOAD_DIR)

if not MODEL_PATH.exists() or not CONFIG_PATH.exists():
    raise FileNotFoundError('msc_model_feedback.py and config_msc_feedback.yaml are required.')

if len(passdata_hits) != 1:
    raise RuntimeError(
        'Expected exactly one CSV whose name contains both "mmc2" and "PASSData"; '
        f'found {[path.name for path in passdata_hits]}'
    )

PASSDATA_UPLOAD = passdata_hits[0]

print('\nModel   :', sha256_file(MODEL_PATH)[:16])
print('Config  :', sha256_file(CONFIG_PATH)[:16])
print('PASSData:', PASSDATA_UPLOAD.name, '|', sha256_file(PASSDATA_UPLOAD)[:16])


In [ ]:
# 0b. Runtime-local output tree and locked constants
ROOT = Path('/content/msc_figure6_tushev/output')

USE_DRIVE_MIRROR = False
DRIVE_MIRROR = Path('/content/drive/MyDrive/MSC_Results_0726/figure6_tushev_harmonised')
if USE_DRIVE_MIRROR:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MIRROR.mkdir(parents=True, exist_ok=True)

INPUT_ROOT = ROOT / 'inputs'
INPUT_DIRS = [INPUT_ROOT / 'paper', INPUT_ROOT / 'supplementary_original', INPUT_ROOT / 'supplementary_csv']
DIRS = {name: ROOT / name for name in [
    'audit', 'annotation', 'manifests', 'panels', 'scores', 'model',
    'checkpoints', 'data', 'statistics', 'plotting', 'reviewer_bundle', 'logs'
]}
for path in [ROOT, *INPUT_DIRS, *DIRS.values()]:
    path.mkdir(parents=True, exist_ok=True)

R_INPUT = ROOT / '10_r_input'
R_INPUT.mkdir(parents=True, exist_ok=True)

# Stage the uploaded PASSData table into the expected input tree.
STAGED_PASSDATA = INPUT_DIRS[2] / PASSDATA_UPLOAD.name
place(PASSDATA_UPLOAD, STAGED_PASSDATA)

DATASET_ID = 'Tushev_2018_CA1_isoform'
ANALYSIS_UNIT = 'isoform_PASS_3pUTR'
SELECTION_SOURCE = 'published_PASSData_localisation_class'
MODEL_VARIANT = 'full_msc_feedback'

FLOAT_FORMAT = '%.17g'
N_MATCHED_PER_GROUP = 250
PANEL_SEED = 20260901
N_BOOT = 20000
N_PERMUTATIONS = 20000
PRIMARY_N_SEEDS = 100
SENSITIVITY_N_SEEDS = 30
CHUNK_SIZE = 10
PARAMETER_SEED_BASE = 30000
SIMULATION_SEED_BASE = 30013
MIN_UTR_LENGTH = 100
MAX_UTR_LENGTH = 5000
TOLERANCE_GRID = (0.025, 0.05, 0.075, 0.10)
MAX_ABS_SMD = 0.1
TRIM_QUANTILE = 0.95
LENGTH_SENSITIVITY_CAPS = (5000, 10000, None)

print('Dataset    :', DATASET_ID)
print('Output root:', ROOT)
print('PASSData   :', STAGED_PASSDATA.relative_to(ROOT))


## 1. Input audit

Every input is hashed. Genuine one-column description files and empty supplementary sheets are accepted during audit.

In [ ]:
def sha256_file(path,chunk_size=1024*1024):
    digest=hashlib.sha256()
    with open(path,'rb') as handle:
        for chunk in iter(lambda:handle.read(chunk_size),b''): digest.update(chunk)
    return digest.hexdigest()

def read_csv_flexible(path):
    attempts=[]; errors=[]
    for encoding in ['utf-8-sig','utf-8','latin-1']:
        for separator in [',','\t',';']:
            try:
                frame=pd.read_csv(path,sep=separator,encoding=encoding,low_memory=False)
                attempts.append((frame.shape[1],encoding,separator,frame))
            except pd.errors.EmptyDataError:
                return pd.DataFrame()
            except Exception as error:
                errors.append(f'{encoding}/{repr(separator)}: {type(error).__name__}: {error}')
    if not attempts: raise RuntimeError(f'Could not parse {path}: '+ ' | '.join(errors))
    return max(attempts,key=lambda item:item[0])[3]

input_files=sorted({path for directory in INPUT_DIRS for path in directory.rglob('*') if path.is_file()})
if not input_files: raise FileNotFoundError(f'No files under {INPUT_ROOT}')
manifest=pd.DataFrame([{'relative_path':str(path.relative_to(INPUT_ROOT)),'filename':path.name,'bytes':path.stat().st_size,'sha256':sha256_file(path)} for path in input_files])
manifest.to_csv(DIRS['manifests']/'input_manifest.csv',index=False)

csv_tables={}
for path in input_files:
    if path.suffix.lower()=='.csv': csv_tables[path.name]=read_csv_flexible(path)

csv_audit=pd.DataFrame([{'filename':name,'rows':len(table),'columns':len(table.columns),'column_names':' | '.join(map(str,table.columns))} for name,table in csv_tables.items()]).sort_values('filename')
csv_audit.to_csv(DIRS['audit']/'csv_table_audit.csv',index=False)

excel_rows=[]
for path in input_files:
    if path.suffix.lower() in {'.xls','.xlsx'}:
        try:
            book=pd.ExcelFile(path)
            for sheet in book.sheet_names:
                preview=pd.read_excel(path,sheet_name=sheet,nrows=5)
                excel_rows.append({'filename':path.name,'sheet':sheet,'preview_rows':len(preview),'preview_columns':len(preview.columns),'column_names':' | '.join(map(str,preview.columns))})
        except Exception as error:
            excel_rows.append({'filename':path.name,'sheet':'READ_ERROR','column_names':repr(error)})
pd.DataFrame(excel_rows).to_csv(DIRS['audit']/'excel_sheet_audit.csv',index=False)
display(csv_audit)
print('Inputs:',len(input_files),'CSV tables:',len(csv_tables),'Excel sheet records:',len(excel_rows))

## 2. Load and validate PASSData

This corrected notebook uses the actual Table S1 columns. `localisation.class` already contains the authors’ edgeR-based localization classification; a separate FDR column is therefore not required.

In [ ]:
pass_matches=[(name,table) for name,table in csv_tables.items() if 'mmc2' in name.lower() and 'passdata' in name.lower()]
if len(pass_matches)!=1: raise RuntimeError(f'Expected one mmc2-PASSData CSV; found {[name for name,_ in pass_matches]}')
PASS_FILENAME,raw=pass_matches[0]
raw=raw.copy(); raw.columns=[str(column).strip() for column in raw.columns]

REQUIRED_COLUMNS=['passid','gene.symbol','gene.feature','upstream.length','rpm.tissue','celltype.class','localisation.class','localisation.logCPM','localisation.logFC']
missing=[column for column in REQUIRED_COLUMNS if column not in raw.columns]
if missing: raise RuntimeError(f'PASSData is missing required columns: {missing}')

profile=[]
for column in raw.columns:
    series=raw[column]; nonnull=series.dropna()
    profile.append({'column':column,'dtype':str(series.dtype),'n_nonnull':int(series.notna().sum()),'n_unique':int(nonnull.nunique()),'examples':' | '.join(nonnull.astype(str).head(8))})
profile=pd.DataFrame(profile)
profile.to_csv(DIRS['audit']/'passdata_column_profile.csv',index=False)
display(profile)

schema={'isoform_id':'passid','gene':'gene.symbol','gene_feature':'gene.feature','utr_length':'upstream.length','expression':'rpm.tissue','cell_type_class':'celltype.class','compartment_class':'localisation.class','compartment_logcpm':'localisation.logCPM','compartment_logfc':'localisation.logFC','strand_source':'passid fwd/rev prefix'}
with open(DIRS['manifests']/'resolved_schema.json','w') as handle: json.dump(schema,handle,indent=2)
print(json.dumps(schema,indent=2))
print('PASSData rows:',len(raw))

## 3. Reconstruct PASS-defined 3′UTR sequences

Table S1 supplies `upstream.length` but not genomic PASS coordinates. The paper defines this value from the closest stop codon to the PASS on rn5. The notebook therefore downloads rn5 and its RefGene annotation, infers strand from `fwdgroup`/`revgroup`, and reconstructs the requested sequence only when a gene has one unique stop boundary on that strand. Ambiguous genes are saved and excluded.

In [ ]:
def numeric(series): return pd.to_numeric(series,errors='coerce')
def clean_sequence(value): return re.sub(r'[^ACGTN]','',str(value).upper().replace('U','T'))
def reverse_complement(sequence): return sequence.translate(str.maketrans('ACGTN','TGCAN'))[::-1]
def download_large(url,destination):
    destination=Path(destination)
    if destination.exists() and destination.stat().st_size>0: print('Reusing',destination.name); return
    temporary=destination.with_name(destination.name+'.part')
    with requests.get(url,stream=True,timeout=(30,1200)) as response:
        response.raise_for_status()
        with open(temporary,'wb') as handle:
            for chunk in response.iter_content(1024*1024):
                if chunk: handle.write(chunk)
    os.replace(temporary,destination); print('Downloaded',destination.name)

pass_text=raw.passid.astype(str).str.lower()
strand=np.select([pass_text.str.startswith('fwd'),pass_text.str.startswith('rev')],['+','-'],default='')
if np.mean(strand!='')<0.95: raise RuntimeError('Could not infer strand from at least 95% of PASS IDs.')

master=pd.DataFrame({'isoform_id':raw.passid.astype(str).str.strip(),'gene':raw['gene.symbol'].astype(str).str.strip(),'gene_feature':raw['gene.feature'].astype(str),'utr_length':numeric(raw['upstream.length']),'expression':numeric(raw['rpm.tissue']),'cell_type_raw':raw['celltype.class'].astype(str),'compartment_raw':raw['localisation.class'].astype(str),'compartment_logcpm':numeric(raw['localisation.logCPM']),'compartment_logfc':numeric(raw['localisation.logFC']),'strand':strand})
if master.isoform_id.duplicated().any(): raise RuntimeError('PASS IDs are not unique.')

RN5_URL='https://hgdownload.soe.ucsc.edu/goldenPath/rn5/bigZips/rn5.fa.gz'
REFGENE_URL='https://hgdownload.soe.ucsc.edu/goldenPath/rn5/database/refGene.txt.gz'
RN5_GZ=DIRS['annotation']/'rn5.fa.gz'; RN5_FA=DIRS['annotation']/'rn5.fa'; REFGENE_GZ=DIRS['annotation']/'rn5_refGene.txt.gz'
download_large(RN5_URL,RN5_GZ); download_large(REFGENE_URL,REFGENE_GZ)
if not RN5_FA.exists():
    temporary=RN5_FA.with_name(RN5_FA.name+'.part')
    with gzip.open(RN5_GZ,'rb') as source,open(temporary,'wb') as destination: shutil.copyfileobj(source,destination)
    os.replace(temporary,RN5_FA)
from pyfaidx import Fasta
genome=Fasta(str(RN5_FA),as_raw=True,sequence_always_upper=True)
chromosomes=set(genome.keys())

columns=['bin','transcript_id','chrom','strand','tx_start','tx_end','cds_start','cds_end','exon_count','exon_starts','exon_ends','score','gene','cds_start_stat','cds_end_stat','exon_frames']
refgene=pd.read_csv(REFGENE_GZ,sep='\t',names=columns,compression='gzip',low_memory=False)
for column in ['tx_start','tx_end','cds_start','cds_end']: refgene[column]=numeric(refgene[column])
refgene=refgene.dropna(subset=['gene','chrom','strand','cds_start','cds_end'])
refgene=refgene[refgene.cds_end>refgene.cds_start].copy()
refgene['stop_boundary']=np.where(refgene.strand=='+',refgene.cds_end,refgene.cds_start).astype(int)
refgene['gene_key']=refgene.gene.astype(str).str.casefold()
lookup={key:group for key,group in refgene.groupby(['gene_key','strand'])}

sequences=[]; statuses=[]; selected_transcripts=[]; selected_chromosomes=[]; selected_boundaries=[]; boundary_counts=[]; audit=[]
for row in master.itertuples(index=False):
    group=lookup.get((row.gene.casefold(),row.strand)); length=row.utr_length
    status=''; sequence=''; transcript=''; chrom=''; boundary=np.nan; n_boundaries=0
    if pd.isna(length) or length<1: status='invalid_upstream_length'
    elif group is None: status='gene_strand_not_in_rn5_refgene'
    else:
        boundaries=group[['chrom','stop_boundary']].drop_duplicates().reset_index(drop=True); n_boundaries=len(boundaries)
        if n_boundaries!=1: status='ambiguous_multiple_stop_boundaries'
        else:
            chrom=str(boundaries.iloc[0].chrom); boundary=int(boundaries.iloc[0].stop_boundary); length=int(round(float(length)))
            if chrom not in chromosomes: status='chromosome_not_in_rn5_fasta'
            elif row.strand=='+':
                left,right=boundary,boundary+length
                if right>len(genome[chrom]): status='interval_out_of_bounds'
                else: sequence=clean_sequence(str(genome[chrom][left:right])); status='reconstructed'
            else:
                left,right=boundary-length,boundary
                if left<0: status='interval_out_of_bounds'
                else: sequence=clean_sequence(reverse_complement(str(genome[chrom][left:right]))); status='reconstructed'
            if status=='reconstructed' and len(sequence)!=length: status='reconstructed_length_mismatch'; sequence=''
            if status=='reconstructed': transcript=group[group.stop_boundary==boundary].sort_values('transcript_id').iloc[0].transcript_id
    sequences.append(sequence); statuses.append(status); selected_transcripts.append(transcript); selected_chromosomes.append(chrom); selected_boundaries.append(boundary); boundary_counts.append(n_boundaries)
    audit.append({'isoform_id':row.isoform_id,'gene':row.gene,'strand':row.strand,'requested_length':length,'n_stop_boundaries':n_boundaries,'status':status,'chrom':chrom,'stop_boundary':boundary})

master['sequence']=sequences; master['sequence_status']=statuses; master['selected_refgene']=selected_transcripts; master['selected_chromosome']=selected_chromosomes; master['selected_stop_boundary']=selected_boundaries; master['stop_boundary_candidates']=boundary_counts
master['sequence_length']=master.sequence.str.len(); master['length_difference']=master.sequence_length-master.utr_length
master['sequence_sha256']=master.sequence.map(lambda x:hashlib.sha256(x.encode()).hexdigest() if x else '')
master['valid_sequence']=(master.sequence_status=='reconstructed')&(master.sequence_length>=MIN_UTR_LENGTH)&(master.sequence_length<=MAX_UTR_LENGTH)&master.sequence.str.fullmatch(r'[ACGTN]+',na=False)&master.length_difference.fillna(0).eq(0)
reconstruction_audit=pd.DataFrame(audit)
reconstruction_audit.to_csv(DIRS['audit']/'rn5_unique_stop_reconstruction_audit.csv',index=False,float_format=FLOAT_FORMAT)
status_table=master.sequence_status.value_counts(dropna=False).rename_axis('status').reset_index(name='n')
status_table.to_csv(DIRS['audit']/'sequence_reconstruction_status.csv',index=False)
master.to_csv(DIRS['data']/'tushev_master_isoform_table.csv.gz',index=False,compression='gzip',float_format=FLOAT_FORMAT)
with open(DIRS['manifests']/'sequence_source.json','w') as handle: json.dump({'source':'rn5 RefGene unique stop boundary plus PASSData upstream.length','valid_sequences':int(master.valid_sequence.sum()),'total_isoforms':len(master),'ambiguous_boundaries_excluded':True,'rn5_fasta_sha256':sha256_file(RN5_GZ),'rn5_refgene_sha256':sha256_file(REFGENE_GZ)},handle,indent=2)
display(status_table)
print('Valid reconstructed sequences:',int(master.valid_sequence.sum()),'of',len(master))
if master.valid_sequence.sum()==0: raise RuntimeError('No valid sequences reconstructed.')

## 4. Published localization classes and sequence-blind eligibility

`localisation.class` contains the published significant class (`neuropil`, `somata`, or `not-localised`). No additional unreported FDR threshold is invented.

In [ ]:
cell_text = master["cell_type_raw"].str.lower()
compartment_text = master["compartment_raw"].str.lower()


# ------------------------------------------------------------
# Identify neuron-enriched isoforms
# ------------------------------------------------------------

master["neuron_enriched"] = (
    cell_text.str.contains(
        "neuron",
        regex=False
    )
    & ~cell_text.str.contains(
        "glia",
        regex=False
    )
    & ~cell_text.str.contains(
        "shared",
        regex=False
    )
)


# ------------------------------------------------------------
# Assign the published compartment-localization classes
# ------------------------------------------------------------

master["localization_group"] = "nonlocalized"

master.loc[
    compartment_text.str.contains(
        "neuropil",
        regex=False
    ),
    "localization_group"
] = "distal_enriched"

master.loc[
    compartment_text.str.contains(
        "somat",
        regex=False
    ),
    "localization_group"
] = "soma_enriched"


# ------------------------------------------------------------
# Restrict the sequence analysis to genuine 3′UTR records
#
# Intronic, coding-sequence, and CDS-extended PASS records
# remain in the master audit table but are excluded from the
# primary sequence and MSC analyses.
# ------------------------------------------------------------

ALLOWED_GENE_FEATURES = {
    "3pUTR",
    "3pUTR_extended"
}

master["valid_3utr_feature"] = (
    master["gene_feature"]
    .isin(ALLOWED_GENE_FEATURES)
)


# ------------------------------------------------------------
# Save feature counts before filtering
# ------------------------------------------------------------

feature_counts_before = (
    master["gene_feature"]
    .value_counts(dropna=False)
    .rename_axis("gene_feature")
    .reset_index(name="n")
)

feature_counts_before.to_csv(
    DIRS["audit"]
    / "gene_feature_counts_before_filter.csv",
    index=False
)


# ------------------------------------------------------------
# Construct the sequence-blind eligible isoform universe
# ------------------------------------------------------------

eligible = master[
    master["neuron_enriched"]
    & master["valid_sequence"]
    & master["valid_3utr_feature"]
    & master["localization_group"].isin(
        [
            "distal_enriched",
            "soma_enriched"
        ]
    )
    & master["expression"].notna()
    & (master["expression"] >= 0)
].copy()


# ------------------------------------------------------------
# Matching covariates and sequence-blind ranking variables
# ------------------------------------------------------------

eligible["log_expression"] = np.log1p(
    eligible["expression"]
)

eligible["log_utr_length"] = np.log(
    eligible["utr_length"]
)

eligible["rank_abs_logfc"] = (
    eligible["compartment_logfc"]
    .abs()
    .fillna(0)
)

eligible["rank_logcpm"] = (
    eligible["compartment_logcpm"]
    .fillna(-np.inf)
)


# ------------------------------------------------------------
# Save feature counts after filtering
# ------------------------------------------------------------

feature_counts_after = (
    eligible
    .groupby(
        [
            "gene_feature",
            "localization_group"
        ]
    )
    .size()
    .rename("n")
    .reset_index()
)

feature_counts_after.to_csv(
    DIRS["audit"]
    / "gene_feature_counts_after_filter.csv",
    index=False
)


# ------------------------------------------------------------
# Full classification audit
# ------------------------------------------------------------

counts = (
    master
    .groupby(
        [
            "neuron_enriched",
            "localization_group",
            "valid_sequence",
            "valid_3utr_feature"
        ]
    )
    .size()
    .rename("n")
    .reset_index()
)

counts.to_csv(
    DIRS["audit"]
    / "classification_counts.csv",
    index=False
)


# ------------------------------------------------------------
# Validate final eligible group sizes
# ------------------------------------------------------------

localization_counts = (
    eligible["localization_group"]
    .value_counts()
    .reindex(
        [
            "distal_enriched",
            "soma_enriched"
        ],
        fill_value=0
    )
)


print("Gene-feature counts before filtering:")
display(feature_counts_before)

print("Eligible feature counts after filtering:")
display(feature_counts_after)

print("Full classification audit:")
display(counts)

print("Eligible genuine 3′UTR localization groups:")
display(localization_counts)


if (
    localization_counts
    < N_MATCHED_PER_GROUP
).any():
    raise RuntimeError(
        "Fewer than 250 eligible genuine 3′UTR "
        "isoforms remain in at least one localization group. "
        "Do not reduce the panel size automatically; inspect "
        "the reconstruction and feature-filter audits."
    )

## 5. Freeze the primary panel

One best-supported isoform per gene and group is retained. Genes occurring in both groups are reserved for the within-gene analysis. The remaining candidates are jointly matched on log tissue expression and log 3′UTR length only.

In [ ]:
def rank_candidates(frame): return frame.sort_values(['rank_abs_logfc','rank_logcpm','expression','isoform_id'],ascending=[False,False,False,True])
best=eligible.groupby(['gene','localization_group'],group_keys=False).apply(rank_candidates).reset_index(drop=True).drop_duplicates(['gene','localization_group'])
sets={name:set(frame.gene) for name,frame in best.groupby('localization_group')}; discordant_genes=sets['distal_enriched']&sets['soma_enriched']
within_candidates=best[best.gene.isin(discordant_genes)].copy(); primary=best[~best.gene.isin(discordant_genes)].copy()
D=rank_candidates(primary[primary.localization_group=='distal_enriched']).reset_index(drop=True); S=rank_candidates(primary[primary.localization_group=='soma_enriched']).reset_index(drop=True)
print('Primary unique-gene candidates:',len(D),len(S),'within-gene discordant genes:',len(discordant_genes))
if len(D)<250 or len(S)<250: raise RuntimeError('Fewer than 250 unique non-discordant genes in a group.')
MATCH=['log_expression','log_utr_length']; combined=pd.concat([D[MATCH],S[MATCH]],ignore_index=True); means=combined.mean(); sds=combined.std(ddof=1)
for variable in MATCH: D[variable+'_z']=(D[variable]-means[variable])/sds[variable]; S[variable+'_z']=(S[variable]-means[variable])/sds[variable]
Z=[x+'_z' for x in MATCH]; DZ=D[Z].to_numpy(float); SZ=S[Z].to_numpy(float); nd,ns=len(D),len(S)
objective=np.r_[((DZ[:,None,:]-SZ[None,:,:])**2).sum(2).min(1),((SZ[:,None,:]-DZ[None,:,:])**2).sum(2).min(1)]+1e-10*np.arange(nd+ns)
def solve(tolerance):
    rows=[np.r_[np.ones(nd),np.zeros(ns)],np.r_[np.zeros(nd),np.ones(ns)]]; lower=[250,250]; upper=[250,250]
    for axis in range(2): rows.append(np.r_[DZ[:,axis],-SZ[:,axis]]); lower.append(-250*tolerance); upper.append(250*tolerance)
    return milp(c=objective,integrality=np.ones(nd+ns,int),bounds=Bounds(np.zeros(nd+ns),np.ones(nd+ns)),constraints=LinearConstraint(np.vstack(rows),np.asarray(lower),np.asarray(upper)),options={'time_limit':300})
selection=None; tolerance_used=None
for tolerance in [0.025,0.05,0.075,0.10]:
    candidate=solve(tolerance); print('Tolerance',tolerance,candidate.message)
    if candidate.success: selection,tolerance_used=candidate,tolerance; break
if selection is None: raise RuntimeError('Sequence-blind matching infeasible through tolerance 0.10.')
chosen=selection.x>0.5; selected_D=D.iloc[np.flatnonzero(chosen[:nd])].copy().reset_index(drop=True); selected_S=S.iloc[np.flatnonzero(chosen[nd:])].copy().reset_index(drop=True)
cost=((selected_D[Z].to_numpy()[:,None,:]-selected_S[Z].to_numpy()[None,:,:])**2).sum(2); di,si=linear_sum_assignment(cost)
final_D=selected_D.iloc[di].copy().reset_index(drop=True); final_S=selected_S.iloc[si].copy().reset_index(drop=True)
final_D['analysis_group']='distal_enriched'; final_S['analysis_group']='soma_enriched_matched'; final_D['matched_soma_isoform']=final_S.isoform_id.to_numpy(); final_S['matched_distal_isoform']=final_D.isoform_id.to_numpy(); final_S['matching_distance']=np.sqrt(cost[di,si])
final_panel=pd.concat([final_D,final_S],ignore_index=True); pairs=pd.DataFrame({'distal_isoform':final_D.isoform_id,'soma_isoform':final_S.isoform_id,'matching_distance':final_S.matching_distance})
if len(final_panel)!=500 or not final_panel.isoform_id.is_unique or not final_panel.gene.is_unique: raise RuntimeError('Frozen panel is not 500 unique isoforms from 500 unique genes.')
def smd(a,b):
    a,b=np.asarray(a,float),np.asarray(b,float); return float((a.mean()-b.mean())/np.sqrt((a.var(ddof=1)+b.var(ddof=1))/2))
balance=pd.DataFrame([{'variable':v,'distal_mean':final_D[v].mean(),'soma_mean':final_S[v].mean(),'standardized_mean_difference':smd(final_D[v],final_S[v]),'paired_spearman_rho':stats.spearmanr(final_D[v],final_S[v]).statistic} for v in MATCH])
if not balance.standardized_mean_difference.abs().lt(0.1).all(): raise RuntimeError('Matching failed absolute SMD < 0.1.')
panel_hash=hashlib.sha256(pd.util.hash_pandas_object(final_panel[['isoform_id','gene','analysis_group','sequence']],index=False).values.tobytes()).hexdigest()
final_panel.to_csv(DIRS['panels']/'tushev_frozen_matched_isoform_panel.csv',index=False,float_format=FLOAT_FORMAT); pairs.to_csv(DIRS['panels']/'tushev_pair_diagnostics.csv',index=False,float_format=FLOAT_FORMAT); balance.to_csv(DIRS['panels']/'tushev_matching_balance.csv',index=False,float_format=FLOAT_FORMAT); within_candidates.to_csv(DIRS['panels']/'tushev_within_gene_candidates.csv',index=False,float_format=FLOAT_FORMAT)
with open(DIRS['manifests']/'panel_manifest.json','w') as handle: json.dump({'dataset':DATASET_ID,'panel_hash':panel_hash,'n_per_group':250,'matching_tolerance':tolerance_used,'matching_variables':MATCH,'discordant_genes_reserved':len(discordant_genes),'sequence_features_used_for_selection':False},handle,indent=2)
print('Panel hash:',panel_hash); display(balance)

### 5b. Support diagnostics and Trim-95 panel

In [ ]:
# 5b. Support diagnostics, reconstruction-filter accounting and Trim-95 panel
support = pd.DataFrame([{
    'dataset': DATASET_ID,
    'group': group,
    'eligible_pool': int(pool_size),
    'selected': N_MATCHED_PER_GROUP,
    'support_exhaustion': N_MATCHED_PER_GROUP / pool_size,
} for group, pool_size in [('distal_enriched', len(D)), ('soma_enriched', len(S))]])

support['matching_tolerance_used'] = tolerance_used
support['discordant_genes_reserved'] = int(len(discordant_genes))
support['within_gene_candidates'] = int(len(within_candidates))
support['min_utr_length'] = MIN_UTR_LENGTH
support['max_utr_length'] = MAX_UTR_LENGTH
support['reconstructed_isoforms'] = int((master.sequence_status == 'reconstructed').sum())
support['excluded_above_max_length'] = int(
    ((master.sequence_status == 'reconstructed') & (master.sequence_length > MAX_UTR_LENGTH)).sum()
)
support['excluded_below_min_length'] = int(
    ((master.sequence_status == 'reconstructed') & (master.sequence_length < MIN_UTR_LENGTH)).sum()
)
support['panel_hash'] = panel_hash
support.to_csv(DIRS['panels'] / 'tushev_support_diagnostics.csv',
               index=False, float_format=FLOAT_FORMAT)

trim_cutoff = float(pairs.matching_distance.quantile(TRIM_QUANTILE))
retained_pairs = pairs[pairs.matching_distance <= trim_cutoff].copy()
trimmed_isoforms = set(retained_pairs.distal_isoform) | set(retained_pairs.soma_isoform)
trimmed_panel = final_panel[final_panel.isoform_id.isin(trimmed_isoforms)].copy()

retained_pairs.to_csv(DIRS['panels'] / 'tushev_retained_pairs_trim95.csv',
                      index=False, float_format=FLOAT_FORMAT)
trimmed_panel.to_csv(DIRS['panels'] / 'tushev_frozen_panel_trim95.csv',
                     index=False, float_format=FLOAT_FORMAT)

print('Trim-95 pairs:', len(retained_pairs), '| cutoff:', round(trim_cutoff, 6))
display(support)


## 6. Sequence scores and paired effects

The score implementation and controls are identical to the current Middleton and Perez workflows.

In [ ]:
def g4_density(sequence,gmin=3,gap=7):
    sequence=clean_sequence(sequence); pattern=re.compile(rf'G{{{gmin},}}[ACGTN]{{1,{gap}}}G{{{gmin},}}[ACGTN]{{1,{gap}}}G{{{gmin},}}[ACGTN]{{1,{gap}}}G{{{gmin},}}'); spans=[match.span() for match in pattern.finditer(sequence)]
    if not spans:return 0.0
    merged=[list(spans[0])]
    for start,end in spans[1:]:
        if start<=merged[-1][1]: merged[-1][1]=max(merged[-1][1],end)
        else: merged.append([start,end])
    return len(merged)*1000/len(sequence)
def feature(sequence):
    sequence=clean_sequence(sequence); length=len(sequence)
    return {'sequence_length':length,'g_fraction':sequence.count('G')/length,'gc_fraction':(sequence.count('G')+sequence.count('C'))/length,'rg4_density':g4_density(sequence,3,7),'rg4_relaxed_density':g4_density(sequence,2,12)}
def build_scores(frame,seed=PANEL_SEED):
    features=pd.DataFrame({row.isoform_id:feature(row.sequence) for row in frame.itertuples()}).T; features.index.name='isoform_id'
    def z(values):
        values=pd.Series(values,index=features.index,dtype=float); return (values-values.mean())/values.std(ddof=1)
    output=pd.DataFrame(index=features.index); output['g_only']=z(features.g_fraction); output['rg4_only']=z(features.rg4_density); output['primary_g_rg4']=(z(features.g_fraction)+z(features.rg4_density))/2
    design=sm.add_constant(pd.DataFrame({'g_fraction':z(features.g_fraction),'log_length':z(np.log(features.sequence_length))},index=features.index)); residual=sm.OLS(z(features.rg4_density),design).fit().resid
    output['rg4_residual']=z(residual); output['relaxed_rg4']=z(features.rg4_relaxed_density); output['gc_control']=z(features.gc_fraction); output['length_control']=z(np.log(features.sequence_length)); rng=np.random.default_rng(seed); output['shuffled_primary']=rng.permutation(output.primary_g_rg4); output['random_score']=rng.normal(size=len(output))
    return features,output
features,scores=build_scores(final_panel); metadata=['analysis_group','gene','expression','utr_length','localization_group','compartment_logfc']; scores=scores.join(final_panel.set_index('isoform_id')[metadata]); features=features.join(final_panel.set_index('isoform_id')[metadata])
features.to_csv(DIRS['scores']/'tushev_primary_panel_sequence_features.csv',float_format=FLOAT_FORMAT); scores.to_csv(DIRS['scores']/'tushev_primary_panel_sequence_scores.csv',float_format=FLOAT_FORMAT)
def infer(values,nboot=N_BOOT,nperm=N_PERMUTATIONS,seed=1):
    values=np.asarray(values,float); rng=np.random.default_rng(seed); observed=values.mean(); bootstrap=values[rng.integers(0,len(values),size=(nboot,len(values)))].mean(1); null=(rng.choice(np.array([-1.,1.]),size=(nperm,len(values)))*values).mean(1)
    return {'n':len(values),'estimate':observed,'median':np.median(values),'sd':values.std(ddof=1),'ci_low':np.percentile(bootstrap,2.5),'ci_high':np.percentile(bootstrap,97.5),'empirical_p':(1+np.sum(np.abs(null)>=abs(observed)))/(nperm+1),'proportion_positive':(values>0).mean()}
score_names=['primary_g_rg4','g_only','rg4_only','rg4_residual','relaxed_rg4','gc_control','length_control','shuffled_primary','random_score']; rows=[]
for index,name in enumerate(score_names):
    distal=scores.loc[pairs.distal_isoform,name].to_numpy(float); soma=scores.loc[pairs.soma_isoform,name].to_numpy(float); result=infer(distal-soma,seed=1000+index); test=stats.mannwhitneyu(distal,soma,alternative='two-sided'); result.update({'dataset':DATASET_ID,'score':name,'mean_distal':distal.mean(),'mean_soma':soma.mean(),'cliffs_delta':2*test.statistic/(len(distal)*len(soma))-1,'mann_whitney_p':test.pvalue}); rows.append(result)
sequence_effects=pd.DataFrame(rows); sequence_effects['q_bh_empirical']=multipletests(sequence_effects.empirical_p,method='fdr_bh')[1]
sequence_effects.to_csv(DIRS['statistics']/'tushev_sequence_effects.csv',index=False,float_format=FLOAT_FORMAT); sequence_effects.to_csv(DIRS['plotting']/'panel_B_tushev_sequence_effects.csv',index=False,float_format=FLOAT_FORMAT); display(sequence_effects)

### 6b. Sparsity and Trim-95 sensitivity

In [ ]:
# 6b. Sequence-feature sparsity and Trim-95 sensitivity for paired effects
sparsity_rows = []
for column, label in [('rg4_density', 'strict_rg4'), ('rg4_relaxed_density', 'relaxed_rg4')]:
    values = features[column].astype(float)
    sparsity_rows.append({
        'dataset': DATASET_ID, 'feature': label, 'n': int(len(values)),
        'zero_fraction': float((values == 0).mean()),
        'median': float(values.median()), 'mean': float(values.mean()),
        'sd': float(values.std(ddof=1)),
    })

g_rg4 = stats.spearmanr(features.g_fraction.astype(float), features.rg4_density.astype(float))
redundancy = stats.spearmanr(scores.primary_g_rg4.astype(float), scores.g_only.astype(float))

sparsity = pd.DataFrame(sparsity_rows)
sparsity['g_vs_rg4_spearman_rho'] = float(g_rg4.statistic)
sparsity['g_vs_rg4_spearman_p'] = float(g_rg4.pvalue)
sparsity['primary_vs_gonly_spearman_rho'] = float(redundancy.statistic)
sparsity['panel_hash'] = panel_hash
sparsity.to_csv(DIRS['statistics'] / 'tushev_sequence_sparsity.csv',
                index=False, float_format=FLOAT_FORMAT)
sparsity.to_csv(DIRS['plotting'] / 'panel_B_tushev_sequence_sparsity.csv',
                index=False, float_format=FLOAT_FORMAT)
display(sparsity)

trim_rows = []
for index, name in enumerate(score_names):
    distal = scores.loc[retained_pairs.distal_isoform, name].to_numpy(float)
    soma = scores.loc[retained_pairs.soma_isoform, name].to_numpy(float)
    result = infer(distal - soma, seed=1500 + index)
    test = stats.mannwhitneyu(distal, soma, alternative='two-sided')
    result.update({'dataset': DATASET_ID, 'score': name, 'analysis': 'trim95_sensitivity',
                   'mean_distal': distal.mean(), 'mean_soma': soma.mean(),
                   'cliffs_delta': 2 * test.statistic / (len(distal) * len(soma)) - 1,
                   'mann_whitney_p': test.pvalue, 'panel_hash': panel_hash})
    trim_rows.append(result)

sequence_effects_trim95 = pd.DataFrame(trim_rows)
sequence_effects_trim95['q_bh_empirical'] = multipletests(
    sequence_effects_trim95.empirical_p, method='fdr_bh')[1]
sequence_effects_trim95.to_csv(DIRS['statistics'] / 'tushev_sequence_effects_trim95.csv',
                               index=False, float_format=FLOAT_FORMAT)
sequence_effects_trim95.to_csv(DIRS['plotting'] / 'panel_B_tushev_sequence_effects_trim95.csv',
                               index=False, float_format=FLOAT_FORMAT)

concordance = (
    sequence_effects[['score', 'estimate', 'empirical_p']]
    .rename(columns={'estimate': 'estimate_full', 'empirical_p': 'p_full'})
    .merge(sequence_effects_trim95[['score', 'estimate', 'empirical_p']]
           .rename(columns={'estimate': 'estimate_trim95', 'empirical_p': 'p_trim95'}), on='score')
)
concordance['sign_agreement'] = (
    np.sign(concordance.estimate_full) == np.sign(concordance.estimate_trim95)
)
concordance.to_csv(DIRS['statistics'] / 'tushev_trim95_concordance.csv',
                   index=False, float_format=FLOAT_FORMAT)
display(concordance)


### 6c. Length-cap sensitivity

In [ ]:
# 6c. Length-cap sensitivity for the 3'UTR reconstruction filter
# The 5000 nt cap is a reconstruction-quality criterion, applied before any sequence
# composition is calculated. This cell repeats the sequence-level analysis under
# alternative caps to show that the cap choice does not drive the reported effects.
# No simulations are re-run; the frozen primary panel is unaffected.
length_rows = []

for cap in LENGTH_SENSITIVITY_CAPS:
    subset = master[
        (master.sequence_status == 'reconstructed')
        & (master.sequence_length >= MIN_UTR_LENGTH)
        & (master.sequence.str.fullmatch(r'[ACGTN]+', na=False))
        & master.length_difference.fillna(0).eq(0)
    ]
    if cap is not None:
        subset = subset[subset.sequence_length <= cap]

    panel_ids = set(final_panel.isoform_id)
    kept = subset[subset.isoform_id.isin(panel_ids)].copy()
    kept_pairs = pairs[
        pairs.distal_isoform.isin(set(kept.isoform_id))
        & pairs.soma_isoform.isin(set(kept.isoform_id))
    ]

    if len(kept_pairs) < 5:
        print('Cap', cap, '-> too few retained pairs:', len(kept_pairs))
        continue

    for index, name in enumerate(score_names):
        distal = scores.loc[kept_pairs.distal_isoform, name].to_numpy(float)
        soma = scores.loc[kept_pairs.soma_isoform, name].to_numpy(float)
        result = infer(distal - soma, seed=1700 + index)
        result.update({'dataset': DATASET_ID, 'score': name,
                       'analysis': 'length_cap_sensitivity',
                       'length_cap': 'none' if cap is None else int(cap),
                       'n_pairs': int(len(kept_pairs)), 'panel_hash': panel_hash})
        length_rows.append(result)

length_sensitivity = pd.DataFrame(length_rows)
if len(length_sensitivity):
    length_sensitivity['q_bh_empirical'] = multipletests(
        length_sensitivity.empirical_p, method='fdr_bh')[1]
    length_sensitivity.to_csv(DIRS['statistics'] / 'tushev_length_cap_sensitivity.csv',
                              index=False, float_format=FLOAT_FORMAT)
    length_sensitivity.to_csv(DIRS['plotting'] / 'panel_B_tushev_length_cap_sensitivity.csv',
                              index=False, float_format=FLOAT_FORMAT)
    display(
        length_sensitivity.pivot_table(index='score', columns='length_cap',
                                       values='estimate', aggfunc='first')
    )


## 7. Within-gene isoform analysis

This supplementary test pairs the best neuropil and somata isoform from each discordant gene, holding gene identity constant.

In [ ]:
within_pairs=[]
for gene,subset in within_candidates.groupby('gene'):
    distal=rank_candidates(subset[subset.localization_group=='distal_enriched']); soma=rank_candidates(subset[subset.localization_group=='soma_enriched'])
    if len(distal) and len(soma): within_pairs.append({'gene':gene,'distal_isoform':distal.iloc[0].isoform_id,'soma_isoform':soma.iloc[0].isoform_id})
within_pairs=pd.DataFrame(within_pairs); within_effects=pd.DataFrame()
if len(within_pairs)>=5:
    ids=set(within_pairs.distal_isoform)|set(within_pairs.soma_isoform); within_frame=eligible[eligible.isoform_id.isin(ids)].copy(); _,within_scores=build_scores(within_frame,seed=PANEL_SEED+1); rows=[]
    for index,name in enumerate(score_names):
        values=within_scores.loc[within_pairs.distal_isoform,name].to_numpy(float)-within_scores.loc[within_pairs.soma_isoform,name].to_numpy(float); result=infer(values,seed=2000+index); result.update({'dataset':DATASET_ID,'analysis':'within_gene_discordant_isoforms','score':name}); rows.append(result)
    within_effects=pd.DataFrame(rows); within_effects['q_bh_empirical']=multipletests(within_effects.empirical_p,method='fdr_bh')[1]
within_pairs.to_csv(DIRS['panels']/'tushev_within_gene_isoform_pairs.csv',index=False); within_effects.to_csv(DIRS['statistics']/'tushev_within_gene_isoform_effects.csv',index=False,float_format=FLOAT_FORMAT)
print('Within-gene pairs:',len(within_pairs)); display(within_effects)

## 8. Load the MSC model and define the WT/ME run plan

In [ ]:
# Model and config were uploaded and verified in section 0.
if str(MODEL_PATH.parent) not in sys.path:
    sys.path.insert(0, str(MODEL_PATH.parent))

import msc_model_feedback as M
importlib.reload(M)
from msc_model_feedback import simulate_one, sample_params, load_config

CFG = dict(load_config(str(CONFIG_PATH)))
ORIGINAL_POOL = M.make_material_pool
assert MODEL_VARIANT in M.MODEL_VARIANTS

shutil.copy2(MODEL_PATH, DIRS['model'] / 'msc_model_feedback_used.py')
shutil.copy2(CONFIG_PATH, DIRS['model'] / 'config_msc_feedback_used.yaml')
M.equation_mapping().to_csv(DIRS['model'] / 'equation_to_code_mapping.csv',
                            index=False, float_format=FLOAT_FORMAT)

JIA = {'E_modulus_kPa': {'ME': 4.7, 'WT': 17.4},
       'tau2_ms': {'ME': 4.8, 'WT': 57.0},
       'mobile_fraction': {'ME': 49.1, 'WT': 16.0}}


def oriented(name):
    return (np.log(JIA[name]['ME'] / JIA[name]['WT']) if name == 'mobile_fraction'
            else np.log(JIA[name]['WT'] / JIA[name]['ME']))


raw_separation = {name: oriented(name) for name in JIA}
mean_raw = float(np.mean(list(raw_separation.values())))
PSD_GAIN = 0.55
separations = {name: PSD_GAIN * value / mean_raw for name, value in raw_separation.items()}

ANALYSES = {'primary_g_rg4': {'score': 'primary_g_rg4', 'mode': 'correct'},
            'g_only': {'score': 'g_only', 'mode': 'correct'},
            'primary_mapping_shuffled': {'score': 'primary_g_rg4', 'mode': 'shuffled'}}
OBSERVABLES = {'tau2_ms': PRIMARY_N_SEEDS,
               'E_modulus_kPa': SENSITIVITY_N_SEEDS,
               'mobile_fraction': SENSITIVITY_N_SEEDS}

RUN_PLAN = [{'observable': observable, 'n_seeds': n, 'analysis_name': analysis,
             'score_definition': spec['score'], 'mode': spec['mode'],
             'psd_state': state, 'anchor': sign * separations[observable]}
            for observable, n in OBSERVABLES.items()
            for analysis, spec in ANALYSES.items()
            for state, sign in [('WT_like', 1), ('ME_like', -1)]]

assert len(RUN_PLAN) == 18

with open(DIRS['manifests'] / 'model_manifest.json', 'w') as handle:
    json.dump({'model_sha256': sha256_file(MODEL_PATH),
               'config_sha256': sha256_file(CONFIG_PATH),
               'model_variant': MODEL_VARIANT,
               'psd_gain': PSD_GAIN,
               'psd_separation': separations,
               'psd_reference_values': JIA,
               'config_keys': list(CFG)}, handle, indent=2)

print('Anchors:', {k: round(float(v), 6) for k, v in separations.items()})
print('Planned simulations:', sum(plan['n_seeds'] for plan in RUN_PLAN))
display(pd.DataFrame(RUN_PLAN))


In [ ]:
_MISSING = object()


def normalize_parameter_name(value):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).lower()
    )


def parameter_value(parameters, names, default=_MISSING):
    """
    Retrieve a model parameter while supporting naming variants such as:
    Ngran / N_gran, Dmaterial / D_material,
    cargoell / cargo_ell, and strongfraction / strong_fraction.
    """

    # First try exact names.
    for name in names:
        if name in parameters:
            return parameters[name]

    # Then try normalized names.
    normalized_parameters = {
        normalize_parameter_name(key): key
        for key in parameters
    }

    for name in names:
        normalized_name = normalize_parameter_name(name)

        if normalized_name in normalized_parameters:
            original_key = normalized_parameters[normalized_name]
            return parameters[original_key]

    if default is not _MISSING:
        return default

    raise KeyError(
        "Could not resolve parameter aliases "
        f"{names}. Available keys are:\n"
        f"{sorted(parameters.keys())}"
    )


def make_pool(coordinates, anchor, mode):
    shifted = (
        np.asarray(coordinates, dtype=float)
        - float(anchor)
    )

    if mode not in {"correct", "shuffled"}:
        raise ValueError(
            f"Unsupported mapping mode: {mode}"
        )

    def pool(parameters, rng):
        dimensions = int(
            parameter_value(
                parameters,
                [
                    "Dmaterial",
                    "D_material",
                    "D"
                ],
                default=1
            )
        )

        n_granules = int(
            parameter_value(
                parameters,
                [
                    "Ngran",
                    "N_gran",
                    "Ngranules",
                    "N_granules"
                ]
            )
        )

        cargo_ell = float(
            parameter_value(
                parameters,
                [
                    "cargoell",
                    "cargo_ell"
                ],
                default=1.0
            )
        )

        strong_fraction = float(
            parameter_value(
                parameters,
                [
                    "strongfraction",
                    "strong_fraction"
                ],
                default=0.3
            )
        )

        if dimensions < 1:
            raise ValueError(
                f"Dmaterial must be at least 1; got {dimensions}"
            )

        if n_granules < 1:
            raise ValueError(
                f"Ngran must be at least 1; got {n_granules}"
            )

        if cargo_ell <= 0:
            raise ValueError(
                f"cargoell must be positive; got {cargo_ell}"
            )

        if not 0 < strong_fraction < 1:
            raise ValueError(
                "strongfraction must lie strictly between 0 and 1; "
                f"got {strong_fraction}"
            )

        transcript_indices = rng.choice(
            len(shifted),
            size=n_granules,
            replace=n_granules > len(shifted)
        )

        true_coordinate = shifted[
            transcript_indices
        ].copy()

        if mode == "correct":
            used_coordinate = true_coordinate.copy()

        else:
            used_coordinate = true_coordinate[
                rng.permutation(n_granules)
            ].copy()

        authentic_material = np.zeros(
            (n_granules, dimensions),
            dtype=float
        )

        model_used_material = np.zeros(
            (n_granules, dimensions),
            dtype=float
        )

        if dimensions > 1:
            shared_dimensions = rng.normal(
                0,
                0.3,
                size=(n_granules, dimensions - 1)
            )

            authentic_material[:, 1:] = (
                shared_dimensions
            )

            model_used_material[:, 1:] = (
                shared_dimensions
            )

        authentic_material[:, 0] = (
            true_coordinate
        )

        model_used_material[:, 0] = (
            used_coordinate
        )

        pool.indices = transcript_indices
        pool.true = true_coordinate
        pool.used = used_coordinate

        dimensions_for_payload = min(
            3,
            dimensions
        )

        authentic_d2 = np.mean(
            authentic_material[
                :, :dimensions_for_payload
            ] ** 2,
            axis=1
        )

        payload = np.exp(
            -0.5
            * authentic_d2
            / cargo_ell ** 2
        )

        threshold = np.quantile(
            payload,
            1.0 - strong_fraction
        )

        strong = payload >= threshold

        return (
            model_used_material,
            payload.astype(float),
            strong.astype(bool)
        )

    pool.indices = None
    pool.true = None
    pool.used = None

    return pool

In [ ]:
parameter_probe = sample_params(
    PARAMETER_SEED_BASE,
    CFG
)

print("Available model parameter keys:")
print(sorted(parameter_probe.keys()))

print("\nResolved injected-pool parameters:")
print(
    "Dmaterial =",
    parameter_value(
        parameter_probe,
        ["Dmaterial", "D_material", "D"],
        default=1
    )
)
print(
    "Ngran =",
    parameter_value(
        parameter_probe,
        ["Ngran", "N_gran", "Ngranules", "N_granules"]
    )
)
print(
    "cargoell =",
    parameter_value(
        parameter_probe,
        ["cargoell", "cargo_ell"],
        default=1.0
    )
)
print(
    "strongfraction =",
    parameter_value(
        parameter_probe,
        ["strongfraction", "strong_fraction"],
        default=0.3
    )
)

In [ ]:
LABELS = final_panel.isoform_id.tolist()

GROUP_MAP = (
    final_panel
    .set_index("isoform_id")
    .analysis_group
    .to_dict()
)

COORDS = {
    name: scores.loc[LABELS, name].to_numpy(float)
    for name in ["primary_g_rg4", "g_only"]
}


# Guard against stale panel/label/score mismatches.

assert len(LABELS) == len(final_panel)
assert set(LABELS) == set(GROUP_MAP)
assert set(LABELS).issubset(set(scores.index))
assert set(final_panel["gene_feature"]).issubset({
    "3pUTR",
    "3pUTR_extended"
})


def run_one(plan, seed_index):
    score_name = plan["score_definition"]

    injected = make_pool(
        COORDS[score_name],
        plan["anchor"],
        plan["mode"]
    )

    M.make_material_pool = injected

    parameter_seed = (
        PARAMETER_SEED_BASE + seed_index
    )

    simulation_seed = (
        SIMULATION_SEED_BASE + seed_index
    )

    try:
        output = simulate_one(
            sample_params(parameter_seed, CFG),
            seed=simulation_seed,
            variant_name=MODEL_VARIANT
        )

    finally:
        M.make_material_pool = ORIGINAL_POOL

    labels = np.asarray([
        LABELS[int(index)]
        for index in injected.indices
    ])

    score_map = scores[score_name].to_dict()

    granules = pd.DataFrame({
        "granule": np.arange(len(labels)),
        "transcript_label": labels,
        "analysis_group": [
            GROUP_MAP[label] for label in labels
        ],
        "source_score": [
            score_map[label] for label in labels
        ],
        "true_coordinate": injected.true,
        "model_used_coordinate": injected.used
    })

    supply = granules.groupby(
        [
            "transcript_label",
            "analysis_group",
            "source_score"
        ],
        as_index=False
    ).agg(
        supply_count=("granule", "size"),
        true_coordinate=("true_coordinate", "mean"),
        model_used_coordinate=(
            "model_used_coordinate",
            "mean"
        )
    )

    docks = output["dock_df"].copy()

    if len(docks):
        lookup = granules.set_index("granule")

        docks["transcript_label"] = (
            docks.granule.map(lookup.transcript_label)
        )

        docks["total_docks"] = 1

        docks["active_docks"] = (
            docks.active_synapse == 1
        ).astype(int)

        docks["total_productive"] = (
            docks.productive == 1
        ).astype(int)

        docks["active_productive"] = (
            (docks.active_synapse == 1)
            & (docks.productive == 1)
        ).astype(int)

        events = docks.groupby(
            "transcript_label",
            as_index=False
        ).agg(
            total_docks=("total_docks", "sum"),
            active_docks=("active_docks", "sum"),
            total_productive=(
                "total_productive",
                "sum"
            ),
            active_productive=(
                "active_productive",
                "sum"
            )
        )

    else:
        events = pd.DataFrame(
            columns=[
                "transcript_label",
                "total_docks",
                "active_docks",
                "total_productive",
                "active_productive"
            ]
        )

    tx = supply.merge(
        events,
        on="transcript_label",
        how="left"
    )

    for column in [
        "total_docks",
        "active_docks",
        "total_productive",
        "active_productive"
    ]:
        tx[column] = (
            tx[column].fillna(0).astype(int)
        )

    meta = {
        "dataset": DATASET_ID,
        "panel_hash": panel_hash,
        "observable": plan["observable"],
        "analysis_name": plan["analysis_name"],
        "score_definition": score_name,
        "mode": plan["mode"],
        "psd_state": plan["psd_state"],
        "anchor": plan["anchor"],
        "seed_index": seed_index,
        "parameter_seed": parameter_seed,
        "simulation_seed": simulation_seed
    }

    for key, value in meta.items():
        tx[key] = value

    summary = pd.DataFrame([{
        **meta,
        "n_granules": int(tx.supply_count.sum()),
        "n_unique_transcripts": len(tx),
        "total_docks": int(tx.total_docks.sum()),
        "active_docks": int(tx.active_docks.sum()),
        "total_productive": int(
            tx.total_productive.sum()
        ),
        "active_productive": int(
            tx.active_productive.sum()
        )
    }])

    return tx, summary


smoke = next(
    plan for plan in RUN_PLAN
    if plan["observable"] == "tau2_ms"
    and plan["analysis_name"] == "primary_g_rg4"
    and plan["psd_state"] == "WT_like"
)

_, smoke_summary = run_one(smoke, 999)

if smoke_summary.total_docks.iloc[0] <= 0:
    raise RuntimeError(
        "Smoke test produced no docking events."
    )

print("Panel hash:", panel_hash)
print("Panel size:", len(final_panel))
display(
    final_panel["gene_feature"]
    .value_counts()
)
display(smoke_summary.T)

## 9. Resumable simulations

Checkpoint filenames include the panel hash, preventing reuse after any panel change.

In [ ]:
def safe(value):return re.sub(r'[^A-Za-z0-9_.-]+','_',str(value))
def checkpoint_paths(plan,start,end):
    stem='__'.join([panel_hash[:12],safe(plan['observable']),safe(plan['analysis_name']),safe(plan['psd_state']),f'{start:03d}-{end-1:03d}']); return DIRS['checkpoints']/f'{stem}__summary.csv',DIRS['checkpoints']/f'{stem}__transcripts.csv.gz'
def atomic_csv(frame,path,compression=None):
    temporary=path.with_name(path.name+'.tmp'); frame.to_csv(temporary,index=False,float_format=FLOAT_FORMAT,compression=compression); os.replace(temporary,path)
def valid_checkpoint(summary_path,transcript_path,seeds):
    if not summary_path.exists() or not transcript_path.exists():return False
    try:
        frame=pd.read_csv(summary_path); return len(frame)==len(seeds) and set(frame.seed_index.astype(int))==set(seeds) and set(frame.panel_hash)=={panel_hash}
    except Exception:return False
for number,plan in enumerate(RUN_PLAN,1):
    print('Condition',number,'/',len(RUN_PLAN),plan)
    for start in range(0,plan['n_seeds'],CHUNK_SIZE):
        end=min(start+CHUNK_SIZE,plan['n_seeds']); seeds=list(range(start,end)); summary_path,transcript_path=checkpoint_paths(plan,start,end)
        if valid_checkpoint(summary_path,transcript_path,seeds):print(' reusiwng',start,'-',end-1);continue
        transcript_frames=[]; summary_frames=[]
        for seed in seeds:
            tx,summary=run_one(plan,seed); transcript_frames.append(tx); summary_frames.append(summary)
        atomic_csv(pd.concat(summary_frames,ignore_index=True),summary_path); atomic_csv(pd.concat(transcript_frames,ignore_index=True),transcript_path,'gzip')
print('Simulation grid complete.')

## 10. Aggregate preferences, WT−ME shifts and mapping contrasts

In [ ]:
summary_paths=sorted(DIRS['checkpoints'].glob(f'{panel_hash[:12]}*__summary.csv')); transcript_paths=sorted(DIRS['checkpoints'].glob(f'{panel_hash[:12]}*__transcripts.csv.gz'))
seed_summary=pd.concat([pd.read_csv(path) for path in summary_paths],ignore_index=True); tx=pd.concat([pd.read_csv(path,compression='gzip') for path in transcript_paths],ignore_index=True); expected=sum(plan['n_seeds'] for plan in RUN_PLAN)
if len(seed_summary)!=expected or seed_summary.duplicated(['observable','analysis_name','psd_state','seed_index']).any():raise RuntimeError('Incomplete or duplicated simulation grid.')
seed_summary.to_csv(DIRS['data']/'tushev_seed_level_counts.csv',index=False,float_format=FLOAT_FORMAT); tx.to_csv(DIRS['data']/'tushev_transcript_seed_counts.csv.gz',index=False,float_format=FLOAT_FORMAT,compression='gzip')
condition=['dataset','panel_hash','observable','analysis_name','score_definition','mode','psd_state','anchor','seed_index','parameter_seed','simulation_seed']; grouped=tx.groupby(condition+['analysis_group'],as_index=False).agg(supply=('supply_count','sum'),physical_docking=('active_docks','sum'),productive_capture=('active_productive','sum')); base=seed_summary[condition].drop_duplicates(); frames=[]
for outcome in ['physical_docking','productive_capture']:
    subset=grouped[condition+['analysis_group','supply',outcome]]; distal=subset[subset.analysis_group=='distal_enriched'].drop(columns='analysis_group').rename(columns={'supply':'distal_supply',outcome:'distal_events'}); soma=subset[subset.analysis_group=='soma_enriched_matched'].drop(columns='analysis_group').rename(columns={'supply':'soma_supply',outcome:'soma_events'}); table=base.merge(distal,on=condition,how='left').merge(soma,on=condition,how='left')
    for column in ['distal_supply','soma_supply','distal_events','soma_events']:table[column]=table[column].fillna(0)
    table['distal_rate']=(table.distal_events+0.5)/(table.distal_supply+1); table['soma_rate']=(table.soma_events+0.5)/(table.soma_supply+1); table['preference_fraction']=table.distal_rate/(table.distal_rate+table.soma_rate); table['log_rate_ratio']=np.log(table.distal_rate/table.soma_rate); table['outcome']=outcome; frames.append(table)
prefs=pd.concat(frames,ignore_index=True); prefs.to_csv(DIRS['data']/'tushev_group_preferences_by_seed.csv',index=False,float_format=FLOAT_FORMAT)
index=['dataset','panel_hash','observable','analysis_name','score_definition','mode','outcome','seed_index','parameter_seed','simulation_seed']; fraction=prefs.pivot_table(index=index,columns='psd_state',values='preference_fraction',aggfunc='first').reset_index(); logratio=prefs.pivot_table(index=index,columns='psd_state',values='log_rate_ratio',aggfunc='first').reset_index()
if fraction[['WT_like','ME_like']].isna().any().any():raise RuntimeError('Incomplete WT/ME pairs.')
shifts=fraction[index].copy(); shifts['WT_preference_fraction']=fraction.WT_like; shifts['ME_preference_fraction']=fraction.ME_like; shifts['shift_preference_fraction']=fraction.WT_like-fraction.ME_like; shifts['WT_log_rate_ratio']=logratio.WT_like; shifts['ME_log_rate_ratio']=logratio.ME_like; shifts['shift_log_rate_ratio']=logratio.WT_like-logratio.ME_like; shifts.to_csv(DIRS['data']/'tushev_wt_me_seed_shifts.csv',index=False,float_format=FLOAT_FORMAT)
rows=[]; group_columns=['dataset','observable','analysis_name','score_definition','mode','outcome']
for key,subset in shifts.groupby(group_columns):
    for metric in ['shift_preference_fraction','shift_log_rate_ratio']:
        result=infer(subset[metric],seed=4000+len(rows));result.update(dict(zip(group_columns,key)));result['metric']=metric;rows.append(result)
shift_summary=pd.DataFrame(rows);shift_summary['q_bh_global']=multipletests(shift_summary.empirical_p,method='fdr_bh')[1]
contrast_rows=[]
for (observable,outcome),subset in shifts.groupby(['observable','outcome']):
    for metric in ['shift_preference_fraction','shift_log_rate_ratio']:
        matrix=subset.pivot_table(index='seed_index',columns='analysis_name',values=metric,aggfunc='first');required=['g_only','primary_g_rg4','primary_mapping_shuffled']
        if matrix[required].isna().any().any():raise RuntimeError('Incomplete mapping contrast matrix.')
        contrast_vectors={'g_only_minus_shuffled':matrix.g_only-matrix.primary_mapping_shuffled,'primary_minus_shuffled':matrix.primary_g_rg4-matrix.primary_mapping_shuffled,'g_only_minus_primary':matrix.g_only-matrix.primary_g_rg4}
        for name,values in contrast_vectors.items():
            result=infer(values,seed=5000+len(contrast_rows));result.update({'dataset':DATASET_ID,'observable':observable,'outcome':outcome,'metric':metric,'contrast':name});contrast_rows.append(result)
mapping_contrasts=pd.DataFrame(contrast_rows);mapping_contrasts['q_bh_global']=multipletests(mapping_contrasts.empirical_p,method='fdr_bh')[1]
state_rows=[]; state_group=['dataset','observable','analysis_name','score_definition','mode','outcome','psd_state']
for key,subset in prefs.groupby(state_group):
    for metric in ['preference_fraction','log_rate_ratio']:
        values=subset[metric].to_numpy(float);rng=np.random.default_rng(6000+len(state_rows));bootstrap=values[rng.integers(0,len(values),size=(N_BOOT,len(values)))].mean(1);state_rows.append({**dict(zip(state_group,key)),'metric':metric,'n':len(values),'estimate':values.mean(),'ci_low':np.percentile(bootstrap,2.5),'ci_high':np.percentile(bootstrap,97.5)})
state_preferences=pd.DataFrame(state_rows)
shift_summary.to_csv(DIRS['statistics']/'tushev_wt_me_shift_summary.csv',index=False,float_format=FLOAT_FORMAT);mapping_contrasts.to_csv(DIRS['statistics']/'tushev_mapping_contrasts.csv',index=False,float_format=FLOAT_FORMAT);state_preferences.to_csv(DIRS['statistics']/'tushev_state_preferences.csv',index=False,float_format=FLOAT_FORMAT)
state_preferences.to_csv(DIRS['plotting']/'panel_C_tushev_state_preferences.csv',index=False,float_format=FLOAT_FORMAT);shift_summary.to_csv(DIRS['plotting']/'panel_D_tushev_wt_me_shifts.csv',index=False,float_format=FLOAT_FORMAT);mapping_contrasts.to_csv(DIRS['plotting']/'panel_D_tushev_mapping_contrasts.csv',index=False,float_format=FLOAT_FORMAT)
display(shift_summary)

## 11. Final audit and bundles

In [ ]:
# 11. Final audit, R inputs, manifest and single reproducibility ZIP
required = [
    DIRS['panels'] / 'tushev_frozen_matched_isoform_panel.csv',
    DIRS['panels'] / 'tushev_pair_diagnostics.csv',
    DIRS['panels'] / 'tushev_matching_balance.csv',
    DIRS['panels'] / 'tushev_support_diagnostics.csv',
    DIRS['scores'] / 'tushev_primary_panel_sequence_features.csv',
    DIRS['scores'] / 'tushev_primary_panel_sequence_scores.csv',
    DIRS['data'] / 'tushev_group_preferences_by_seed.csv',
    DIRS['data'] / 'tushev_wt_me_seed_shifts.csv',
    DIRS['plotting'] / 'panel_B_tushev_sequence_effects.csv',
    DIRS['plotting'] / 'panel_C_tushev_state_preferences.csv',
    DIRS['plotting'] / 'panel_D_tushev_wt_me_shifts.csv',
    DIRS['plotting'] / 'panel_D_tushev_mapping_contrasts.csv',
]

coverage = seed_summary.groupby(['observable', 'analysis_name', 'psd_state']).seed_index.nunique().to_dict()
expected_coverage = {(observable, analysis, state): n
                     for observable, n in OBSERVABLES.items()
                     for analysis in ANALYSES
                     for state in ['WT_like', 'ME_like']}
coverage_ok = all(int(coverage.get(key, -1)) == int(value) for key, value in expected_coverage.items())

finite_ok = all(
    np.isfinite(frame[columns].to_numpy(float)).all()
    for frame, columns in [
        (prefs, ['preference_fraction', 'log_rate_ratio']),
        (shifts, ['shift_preference_fraction', 'shift_log_rate_ratio']),
        (shift_summary, ['estimate', 'ci_low', 'ci_high', 'empirical_p']),
        (mapping_contrasts, ['estimate', 'ci_low', 'ci_high', 'empirical_p']),
    ]
)

outputs_ok = all(path.exists() and path.stat().st_size > 0 for path in required)
balance_ok = bool(balance.standardized_mean_difference.abs().lt(MAX_ABS_SMD).all())

# Canonical R tables come only from plotting/.
canonical_names = [
    'panel_B_tushev_sequence_effects.csv',
    'panel_C_tushev_state_preferences.csv',
    'panel_D_tushev_wt_me_shifts.csv',
    'panel_D_tushev_mapping_contrasts.csv',
]
found = {}
for name in canonical_names:
    path = DIRS['plotting'] / name
    assert path.exists() and path.stat().st_size > 0, f'Canonical table missing or empty: {path}'
    found[name] = path

sequence_effects_final = pd.read_csv(found['panel_B_tushev_sequence_effects.csv'])
state_preferences_final = pd.read_csv(found['panel_C_tushev_state_preferences.csv'])
wt_me_shifts_final = pd.read_csv(found['panel_D_tushev_wt_me_shifts.csv'])
mapping_contrasts_final = pd.read_csv(found['panel_D_tushev_mapping_contrasts.csv'])

assert set(sequence_effects_final['dataset']) == {DATASET_ID}
assert len(sequence_effects_final) == len(score_names)
assert set(sequence_effects_final['n']) == {N_MATCHED_PER_GROUP}
assert set(state_preferences_final['n_seeds']) == {SENSITIVITY_N_SEEDS, PRIMARY_N_SEEDS}
assert set(wt_me_shifts_final['n']) == {SENSITIVITY_N_SEEDS, PRIMARY_N_SEEDS}
assert set(mapping_contrasts_final['n']) == {SENSITIVITY_N_SEEDS, PRIMARY_N_SEEDS}
assert {'primary_g_rg4', 'g_only', 'primary_mapping_shuffled'}.issubset(
    set(wt_me_shifts_final['analysis_name']))

for name, path in found.items():
    shutil.copy2(path, R_INPUT / name)

supplementary_inputs = [
    (DIRS['panels'], 'tushev_frozen_matched_isoform_panel.csv'),
    (DIRS['panels'], 'tushev_matching_balance.csv'),
    (DIRS['panels'], 'tushev_pair_diagnostics.csv'),
    (DIRS['panels'], 'tushev_support_diagnostics.csv'),
    (DIRS['panels'], 'tushev_retained_pairs_trim95.csv'),
    (DIRS['panels'], 'tushev_frozen_panel_trim95.csv'),
    (DIRS['panels'], 'tushev_within_gene_isoform_pairs.csv'),
    (DIRS['statistics'], 'tushev_sequence_sparsity.csv'),
    (DIRS['statistics'], 'tushev_sequence_effects_trim95.csv'),
    (DIRS['statistics'], 'tushev_trim95_concordance.csv'),
    (DIRS['statistics'], 'tushev_length_cap_sensitivity.csv'),
    (DIRS['statistics'], 'tushev_within_gene_isoform_effects.csv'),
]
for directory, name in supplementary_inputs:
    path = directory / name
    if path.exists() and path.stat().st_size > 0:
        shutil.copy2(path, R_INPUT / name)
        print('Copied:', name)
    else:
        print('Missing or empty supplementary table:', name)

if not all([coverage_ok, finite_ok, outputs_ok, balance_ok]):
    raise RuntimeError(
        f'FINAL AUDIT FAILED: coverage={coverage_ok} finite={finite_ok} '
        f'outputs={outputs_ok} balance={balance_ok}'
    )

manifest = {
    'dataset': DATASET_ID,
    'analysis_unit': ANALYSIS_UNIT,
    'group_assignment': SELECTION_SOURCE,
    'panel_hash': panel_hash,
    'canonical_source_directory': str(DIRS['plotting']),
    'passdata_source': PASSDATA_UPLOAD.name,
    'passdata_sha256': sha256_file(STAGED_PASSDATA),
    'genome_build': 'rn5 (UCSC)',
    'model_sha256': sha256_file(MODEL_PATH),
    'config_sha256': sha256_file(CONFIG_PATH),
    'model_variant': MODEL_VARIANT,
    'psd_gain': PSD_GAIN,
    'psd_separation': separations,
    'psd_reference_values': JIA,
    'invariant_layer': {
        'n_matched_per_group': N_MATCHED_PER_GROUP,
        'tolerance_grid': list(TOLERANCE_GRID),
        'matching_tolerance_used': tolerance_used,
        'max_abs_smd': MAX_ABS_SMD,
        'score_names': score_names,
        'n_boot': N_BOOT,
        'n_permutations': N_PERMUTATIONS,
        'primary_n_seeds': PRIMARY_N_SEEDS,
        'sensitivity_n_seeds': SENSITIVITY_N_SEEDS,
        'parameter_seed_base': PARAMETER_SEED_BASE,
        'simulation_seed_base': SIMULATION_SEED_BASE,
        'trim_quantile': TRIM_QUANTILE,
    },
    'dataset_specific_layer': {
        'min_utr_length': MIN_UTR_LENGTH,
        'max_utr_length': MAX_UTR_LENGTH,
        'length_sensitivity_caps': ['none' if cap is None else int(cap)
                                    for cap in LENGTH_SENSITIVITY_CAPS],
        'allowed_gene_features': sorted(ALLOWED_GENE_FEATURES),
        'within_gene_pairs': int(len(within_pairs)),
        'discordant_genes_reserved': int(len(discordant_genes)),
    },
    'validation': {
        'coverage_ok': bool(coverage_ok),
        'finite_ok': bool(finite_ok),
        'outputs_ok': bool(outputs_ok),
        'balance_ok': bool(balance_ok),
        'expected_simulations': int(expected),
        'observed_simulations': int(len(seed_summary)),
        'panel_size': int(len(final_panel)),
        'trim95_pairs': int(len(retained_pairs)),
        'sequence_effect_rows': int(len(sequence_effects_final)),
        'state_seed_counts': sorted(state_preferences_final['n_seeds'].unique().tolist()),
        'shift_seed_counts': sorted(wt_me_shifts_final['n'].unique().tolist()),
        'canonical_tables': canonical_names,
    },
    'completed_at': datetime.datetime.now().isoformat(),
    'outputs': {str(path.relative_to(ROOT)): sha256_file(path)
                for path in sorted(ROOT.rglob('*')) if path.is_file()},
}
(ROOT / 'MANIFEST_tushev.json').write_text(json.dumps(manifest, indent=2))

anchor_text = {key: round(float(value), 6) for key, value in separations.items()}

readme = f"""# MSC Figure 6 Tushev reproducibility bundle

Dataset: Tushev et al. 2018, CA1 PASS isoform data, published localisation classes
from supplementary table mmc2 (PASSData). Sequences are reconstructed from the
UCSC rn5 assembly and rn5 refGene annotation. Analysis unit: PASS-defined 3'UTR
isoform, restricted to gene features {sorted(ALLOWED_GENE_FEATURES)}.

Group assignment uses the published localisation classes: neuropil records become
distal_enriched, somatic records become soma_enriched. Group assignment and
matching are sequence-blind. The frozen panel contains {N_MATCHED_PER_GROUP}
distal and {N_MATCHED_PER_GROUP} matched soma isoforms from 500 distinct genes.
Genes with discordant isoforms are reserved for the within-gene analysis and
excluded from the primary panel.

Invariant analysis layer, shared with the Middleton and Perez workflows:
{N_MATCHED_PER_GROUP} per group, matching over the tolerance ladder
{list(TOLERANCE_GRID)} with absolute standardised mean difference below
{MAX_ABS_SMD}, nine score definitions, {N_BOOT} bootstrap resamples,
{N_PERMUTATIONS} sign-flip permutations with Benjamini-Hochberg correction, and an
18-condition simulation grid using {PRIMARY_N_SEEDS} seeds for the primary tau2
PSD mapping and {SENSITIVITY_N_SEEDS} seeds each for elastic modulus and mobile
fraction. PSD anchors use gain {PSD_GAIN}, giving {anchor_text}.

Dataset-specific layer: sequences are accepted only if reconstruction succeeded,
contains only ACGTN, the reconstructed length matches the PASS annotation exactly,
and the length lies between {MIN_UTR_LENGTH} and {MAX_UTR_LENGTH} nucleotides. The
upper bound is a reconstruction-quality criterion applied before any sequence
composition is computed. A length-cap sensitivity analysis repeats the paired
sequence-effect inference under alternative caps.

Simulations call msc_model_feedback.simulate_one with variant {MODEL_VARIANT}. The
isoform-informed material pool temporarily replaces make_material_pool. Docking
receives the model-used material coordinate while the productive payload follows
the authentic isoform coordinate, so true and used coordinates stay separable and
the shuffled-mapping control is not tautological.

Checkpoints are keyed by the panel hash {panel_hash[:12]}.

The within-gene isoform analysis is reported as found: {len(within_pairs)}
discordant isoform pair(s) were available, and the paired test requires at least
five, so it is not evaluable in this dataset.

Canonical tables consumed by the three-dataset plotting workflow are copied from
plotting/ into 10_r_input/.

All docking and productive-capture quantities are stochastic in-silico predictions
of the MSC model, not wet-lab measurements.
"""
(ROOT / 'README.md').write_text(readme)

plotting_zip = shutil.make_archive(str(ROOT / 'tushev_R_plotting_tables'), 'zip',
                                   root_dir=DIRS['plotting'])
reviewer_root = DIRS['reviewer_bundle']
for name in ['audit', 'manifests', 'panels', 'scores', 'statistics', 'plotting']:
    destination = reviewer_root / name
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(DIRS[name], destination)
reviewer_zip = shutil.make_archive(str(ROOT / 'tushev_reviewer_bundle'), 'zip',
                                   root_dir=reviewer_root)

zip_path = Path('/content/MSC_Figure6_Tushev_reproducible_bundle.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ROOT.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(ROOT.parent))

assert zip_path.exists() and zip_path.stat().st_size > 0
if USE_DRIVE_MIRROR:
    shutil.copy2(zip_path, DRIVE_MIRROR / zip_path.name)

print('\nFINAL AUDIT PASSED')
print('Panel hash:', panel_hash)
print('Within-gene pairs:', len(within_pairs))
print('ZIP:', zip_path.name, f'{zip_path.stat().st_size / 1e6:.1f} MB')
print('\nCanonical R inputs:')
for name in canonical_names:
    print('-', R_INPUT / name)
display(sequence_effects_final)
files.download(str(zip_path))
